In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
import scienceplots
plt.style.use(['science', 'nature', 'grid'])

import numpy as np
from scipy.interpolate import interp1d
import pandas as pd

from LoadMoments import *
from LoadTSV import *

In [ ]:
# mpl.use('pgf')
mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 10,        # Match your LaTeX document font size
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8
})

In [ ]:
# Plotting
colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown']
linestyles = ['-', '--', ':', '-.', (0, (3, 1, 1, 1)), (0, (5, 1))]

# Parameter
M = 4
closure = "ExtGram"
Kn = 1.0
source_string = "relaxation_source"
T_end = 0.3
base_tree_level = 10
polydeg = 1
rho_L = 7.0
v_L1 = 0.0
v_L2 = 0.0
v_L3 = 0.0
theta_L = 1.0
rho_R = 1.0
v_R1 = 0.0
v_R2 = 0.0
v_R3 = 0.0
theta_R = 1.0

marker = ['o', 's', '^', 'x', 'd']
angle_indizes = [1, 2, 3]

# Kn and relaxation for solution files
M_vec = [4, 6, 8]
source = 'relaxation_source'
name = lambda M, angleindex, Kn, source_string: f"../out/1D3V_full/1D3V_M{M}_closure{closure}_Kn{Kn}_source{source_string}_angleindex{angleindex}_cons.tsv"

In [ ]:
for M in M_vec:
    fig_shared, axs_shared = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(6, 3))
    fig_shared.suptitle(f'1D-3V Full: M={M}, all angle indices', fontsize=12)

    twx_shared = [ax.twinx() for ax in axs_shared]
    for twx in twx_shared[1:]:
        twx.sharey(twx_shared[0])
    for twx in twx_shared:
        twx.grid(False)
    for twx in twx_shared[:-1]:
        twx.tick_params(labelright=False)      # right tick labels only on the last column

    for ax in axs_shared:
        ax.set_xlim(-1, 1)
        ax.set_xlabel('x')
    axs_shared[0].set_ylabel(r'$\rho, p$')
    twx_shared[-1].set_ylabel('$v$')

    for color, label in zip(colors, [r'$\rho$', r'$v$', r'$p$']):
        axs_shared[0].plot([], [], color=color, label=label)


    # 3. Data Loop with Row and Column Titles
    for row, angle_index in enumerate(angle_indizes):
        kn_titles = [r'Kn=0.1', r'Kn=1.0', r'Kn$\to \infty$']

        for column, (Kn, source) in enumerate([(0.1, 'relaxation_source'), (1.0, 'relaxation_source'), (10.0, 'zero_source')]):
            print(f"Processing M={M}, Kn={Kn}, source={source}, angle_pair={angle_index}")
            solution_file = name(M, angle_index, Kn, source)
            data = read_solution_file(solution_file)
            
            times = np.array([data[ts]["t"] for ts in sorted(data.keys())])
            timesteps = np.array(sorted(data.keys()))

            if len(timesteps) != 11: # Solution diverged before
                print(f"Warning: Expected 11 timesteps for M={M}, Kn={Kn}, source={source}, angle_pair={angle_index}, but got {len(timesteps)}. Skipping this case.")
                continue  # Skip this iteration if the number of timesteps is not as expected
            
            time_query = 21 
            sol = interpolate_solution(data, time_query, times, timesteps)
            x, U = sol["x"], sol["u"]

            rho = U[:, 0]
            v = U[:, 1] / U[:, 0] # vx
            theta = 1 / (3 * rho) * (U[:, 4] + U[:, 7] + U[:, 9] - rho * v**2) 
            p = rho * theta

            n_markers = 8                                   # per curve, tune to taste
            step = max(len(x) // n_markers, 1)
            offset = row * step // len(angle_indizes)       # 0, step/3, 2*step/3
            style = dict(linestyle=linestyles[row], marker=marker[row],
                        markevery=(offset, step), markersize=4, markerfacecolor='none')

            axs_shared[column].plot(x, rho, color=colors[0], **style)
            twx_shared[column].plot(x, v,   color=colors[1], **style)
            axs_shared[column].plot(x, p,   color=colors[2], **style)

    axs_shared[0].plot([], [], color='black', linestyle=linestyles[0], label=r'Angle Index 1', marker=marker[0])
    axs_shared[0].plot([], [], color='black', linestyle=linestyles[1], label=r'Angle Index 2', marker=marker[1])
    axs_shared[0].plot([], [], color='black', linestyle=linestyles[2], label=r'Angle Index 3', marker=marker[2])

    fig_shared.legend(loc='lower center', ncol=7)
    fig_shared.tight_layout(rect=(0, 0.05, 1, 1))
    fig_shared.savefig(f'../out/Figures/1D3V/full/AngleComparison_M{M}_shared.pdf')
    fig_shared.show()

In [ ]:
angle_index = 1
fig, axs = plt.subplots(2, 2, figsize=(6.8, 5), sharex=True, sharey=False)

for ax, label in zip(axs[:, 0], ['Kn=0.1', 'Kn=1.0']): # ToDo: 0.2
    ax.set_ylabel("$\\rho$")
    ax.text(-0.25, 0.5, label, transform=ax.transAxes, rotation=90, va='center', ha='center', size='large', weight='bold')
for ax in axs[:, 1]:
    ax.set_ylabel("$p$")

for row, (Kn, source) in enumerate([(0.1, 'relaxation_source'), (1.0, 'relaxation_source')]): # ToDo: 0.2
    twx1 = axs[row, 0].twinx(); twx1.grid(False); twx1.set_ylabel("$v$")
    twx2 = axs[row, 1].twinx(); twx2.grid(False); twx2.set_ylabel("$\\theta$")
    for M_index, M in enumerate([4, 6, 8]):
        solution_file = name(M, angle_index, Kn, source)
        data = read_solution_file(solution_file)
        
        times = np.array([data[ts]["t"] for ts in sorted(data.keys())])
        timesteps = np.array(sorted(data.keys()))

        if len(timesteps) != 11: # Solution diverged before
            print(f"Warning: Expected 11 timesteps for M={M}, Kn={Kn}, source={source}, angle_index={angle_index}, but got {len(timesteps)}. Skipping this case.")
            continue  # Skip this iteration if the number of timesteps is not as expected
        
        time_query = 21 
        sol = interpolate_solution(data, time_query, times, timesteps)
        x, U = sol["x"], sol["u"]

        rho = U[:, 0]
        v = U[:, 1] / U[:, 0] # vx
        theta = 1 / (3 * rho) * (U[:, 4] + U[:, 7] + U[:, 9] - rho * v**2) 
        p = rho * theta

        axs[row, 0].plot(x, rho, color=colors[0], linestyle=linestyles[M_index])
        twx1.plot(x, v, color=colors[1], linestyle=linestyles[M_index])
        
        axs[row, 1].plot(x, p, color=colors[2], linestyle=linestyles[M_index])
        twx2.plot(x, theta, color=colors[3], linestyle=linestyles[M_index])

[ax.set_xlim(-1, 1) for ax in axs.flatten()]
axs[-1, 0].set_xlabel("x"); axs[-1, 1].set_xlabel("x")
[axs[0, 0].plot([], [], color='black', linestyle=linestyles[M_index], lw=1, label=f'M={M}') for M_index, M in enumerate([4, 6, 8])]
[axs[0, 0].plot([], [], color=color, label=label) for color, label in zip(colors, [r'$\rho$', r'$v$', r'$p$', r'$\theta$'])]

handles, labels = axs[0, 0].get_legend_handles_labels()
x_left, x_right = axs[0, 0].get_position().x0, axs[0, 1].get_position().x1
legend_ncol = 7
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=((x_left + x_right) / 2, 1.0), ncol=legend_ncol, frameon=False)

fig.tight_layout()
# fig.savefig(f"../out/Figures/1D3V/slab/KnComparison_rhoL_{rhoL}.pdf", bbox_inches='tight')
fig.show()

In [ ]:
for M in M_vec:
    fig_shared, axs_shared = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(6, 3))
    fig_shared.suptitle(f'1D-3V Full: M={M}, all angle indices', fontsize=12)

    twx_shared = [ax.twinx() for ax in axs_shared]
    for twx in twx_shared:
        twx.grid(False)
    for twx in twx_shared[:-1]:
        twx.tick_params(labelright=False)      # right tick labels only on the last column

    for ax in axs_shared:
        ax.set_xlim(-1, 1)
        ax.set_xlabel('x')
    axs_shared[0].set_ylabel(r'$\rho, p$')
    twx_shared[-1].set_ylabel('$v$')

    for color, label in zip(colors, [r'Last Moment', r'Second Last Moment']):
        axs_shared[0].plot([], [], color=color, label=label)


    # 3. Data Loop with Row and Column Titles
    for row, angle_index in enumerate(angle_indizes):
        twx_dict = {}
        first_twx = None
        kn_titles = [r'Kn=0.1', r'Kn=1.0', r'Kn$\to \infty$']

        for column, (Kn, source) in enumerate([(0.1, 'relaxation_source'), (1.0, 'relaxation_source'), (10.0, 'zero_source')]):
            # Set Column Titles on the top row only
            if row == 0:
                ax.set_title(kn_titles[column])

            print(f"Processing M={M}, Kn={Kn}, source={source}, angle_pair={angle_index}")
            solution_file = name(angle_index, Kn, source)
            data = read_solution_file(solution_file)
            
            times = np.array([data[ts]["t"] for ts in sorted(data.keys())])
            timesteps = np.array(sorted(data.keys()))

            if len(timesteps) != 11: # Solution diverged before
                print(f"Warning: Expected 11 timesteps for M={M}, Kn={Kn}, source={source}, angle_pair={angle_index}, but got {len(timesteps)}. Skipping this case.")
                continue  # Skip this iteration if the number of timesteps is not as expected
            
            time_query = 21 
            sol = interpolate_solution(data, time_query, times, timesteps)
            x, U = sol["x"], sol["u"]

            n_markers = 8                                   # per curve, tune to taste
            step = max(len(x) // n_markers, 1)
            offset = row * step // len(angle_indizes)       # 0, step/3, 2*step/3
            style = dict(linestyle=linestyles[row], marker=marker[row],
                        markevery=(offset, step), markersize=4, markerfacecolor='none')

            axs_shared[column].plot(x, U[:, -1], color=colors[0], **style)
            twx_shared[column].plot(x, U[:, -2],   color=colors[1], **style)

    axs_shared[0].plot([], [], color='black', linestyle=linestyles[0], label=r'Angle Index 1', marker=marker[0])
    axs_shared[0].plot([], [], color='black', linestyle=linestyles[1], label=r'Angle Index 2', marker=marker[1])
    axs_shared[0].plot([], [], color='black', linestyle=linestyles[2], label=r'Angle Index 3', marker=marker[2])

    fig_shared.legend(loc='lower center', ncol=7)
    fig_shared.tight_layout(rect=(0, 0.05, 1, 1))
    fig_shared.savefig(f'../out/Figures/1D3V/full/AngleComparison_M{M}_LastMoments.pdf')
    fig_shared.show()